# Portfolio edition

Historical market-gap notebook using the July 28 snapshot: 791 seed listings (680 competitors and 111 Hiep Thuy), grouped into 17 categories. This differs from the later 794-listing / 16-group reporting snapshot.

Source code retained; cell outputs and execution metadata cleared for publication. See RUNNING.md for inputs, historical validation and limitations.


# MIS495 — Phân tích thị trường hạt giống và khoảng trống Hiệp Thủy

Notebook Google Colab tái tạo toàn bộ file `MIS495_Seed_Market_Gap_Analysis.xlsx` từ CSV sản phẩm Shopee.

## Mục tiêu kiểm chứng

Sau khi chọn **Runtime → Run all**, notebook sẽ:

1. Đọc đúng bốn CSV sản phẩm.
2. Chỉ dùng **Hà Bắc, Quang Anh VNUA và Vườn Babylon** làm benchmark nhu cầu/giá.
3. Chỉ dùng **Hiệp Thủy** để đo khoảng trống danh mục; **An Việt bị loại hoàn toàn**.
4. Phân loại listing hạt giống, tính P25–P75, sold count, review, chỉ số nhu cầu và điểm cơ hội.
5. Xuất workbook năm sheet giống file kết quả đã nộp.
6. Chạy các kiểm tra QA và, nếu có file Excel tham chiếu, so sánh từng ô dữ liệu.

### Kết quả kỳ vọng với snapshot đã dùng trong báo cáo

- 680 listing hạt giống từ ba shop benchmark.
- 111 listing hạt giống Hiệp Thủy.
- 17 nhóm thị trường, 50 cụm sản phẩm và 791 SKU đã phân loại.
- 0 dòng benchmark chưa phân loại, 0 duplicate `shop + item_id`, 0 listing đối thủ thiếu giá.


## 1. Setup

Cell này chỉ cài các package còn thiếu. Các thư viện đều công khai và chạy được trên Google Colab.


In [ ]:
import importlib.util
import subprocess
import sys

required_packages = {
    "pandas": "pandas>=2.0",
    "numpy": "numpy>=1.24",
    "xlsxwriter": "XlsxWriter>=3.1",
    "openpyxl": "openpyxl>=3.1",
}
missing = [pip_name for module, pip_name in required_packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

import json
import math
import os
import shutil
import warnings
import zipfile
from datetime import date, datetime
from pathlib import Path

import numpy as np
import openpyxl
import pandas as pd
import xlsxwriter
from IPython.display import display

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("XlsxWriter:", xlsxwriter.__version__)
print("openpyxl:", openpyxl.__version__)


## 2. Chọn dữ liệu đầu vào

Trên Colab, upload một trong hai dạng:

- Một file ZIP có chứa bốn CSV; hoặc
- Bốn CSV riêng lẻ.

Có thể upload thêm file Excel kết quả đã nộp để cell kiểm tra cuối đối chiếu tự động.

Tên bốn CSV bắt buộc:

```text
gia san pham - Ha Bac.csv
gia san pham - Quang Anh VNUA.csv
gia san pham - Vuon Babylon.csv
gia san pham - Hiep Thuy (shop minh).csv
```


In [ ]:
ANALYSIS_DATE = "2026-07-28"  # ngày của snapshot trong file Excel đã nộp
OUTPUT_FILENAME = "MIS495_Seed_Market_Gap_Analysis.xlsx"
REQUIRED_FILENAMES = [
    "gia san pham - Ha Bac.csv",
    "gia san pham - Quang Anh VNUA.csv",
    "gia san pham - Vuon Babylon.csv",
    "gia san pham - Hiep Thuy (shop minh).csv",
]

in_colab = False
try:
    from google.colab import files as colab_files
    in_colab = True
except ImportError:
    colab_files = None

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
default_root = Path("/content") / f"mis495_seed_market_{timestamp}" if in_colab else Path.cwd() / f"mis495_seed_market_{timestamp}"
RUN_ROOT = Path(os.getenv("MIS495_RUN_ROOT", str(default_root))).expanduser().resolve()
OUTPUT_DIR = RUN_ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def safe_extract(zip_path: Path, destination: Path) -> None:
    destination.mkdir(parents=True, exist_ok=True)
    destination_resolved = destination.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if destination_resolved not in target.parents and target != destination_resolved:
                raise ValueError(f"ZIP có đường dẫn không an toàn: {member.filename}")
        archive.extractall(destination)


def find_required_files(search_root: Path) -> dict[str, Path]:
    matches = {}
    for required_name in REQUIRED_FILENAMES:
        candidates = sorted(search_root.rglob(required_name))
        if not candidates:
            raise FileNotFoundError(f"Không tìm thấy file bắt buộc: {required_name}")
        matches[required_name] = candidates[0]
    return matches


reference_xlsx = None
local_input = os.getenv("MIS495_INPUT_DIR")
if local_input:
    INPUT_DIR = Path(local_input).expanduser().resolve()
    missing_names = [name for name in REQUIRED_FILENAMES if not (INPUT_DIR / name).is_file()]
    if missing_names:
        raise FileNotFoundError(f"Folder local thiếu file: {missing_names}")
    reference_env = os.getenv("MIS495_REFERENCE_XLSX")
    if reference_env:
        reference_xlsx = Path(reference_env).expanduser().resolve()
else:
    if not in_colab:
        raise RuntimeError("Chạy local cần đặt biến MIS495_INPUT_DIR. Trên Colab, cell này sẽ mở hộp upload.")
    print("Hãy chọn ZIP hoặc bốn CSV. Có thể chọn thêm file Excel tham chiếu trong cùng lượt upload.")
    uploaded = colab_files.upload()
    upload_root = RUN_ROOT / "uploaded"
    upload_root.mkdir(parents=True, exist_ok=True)
    for name, content in uploaded.items():
        (upload_root / Path(name).name).write_bytes(content)
    for zip_path in upload_root.glob("*.zip"):
        safe_extract(zip_path, upload_root / f"unzipped_{zip_path.stem}")
    found = find_required_files(upload_root)
    INPUT_DIR = RUN_ROOT / "input_csv"
    INPUT_DIR.mkdir(parents=True, exist_ok=True)
    for required_name, source_path in found.items():
        shutil.copy2(source_path, INPUT_DIR / required_name)
    xlsx_candidates = sorted(upload_root.rglob("*.xlsx"))
    if xlsx_candidates:
        reference_xlsx = xlsx_candidates[0]

input_audit = pd.DataFrame(
    [
        {
            "file": name,
            "path": str(INPUT_DIR / name),
            "size_kb": round((INPUT_DIR / name).stat().st_size / 1024, 1),
        }
        for name in REQUIRED_FILENAMES
    ]
)
display(input_audit)
print("Input directory:", INPUT_DIR)
print("Reference workbook:", reference_xlsx if reference_xlsx else "Không upload — sẽ bỏ qua đối chiếu từng ô")


## 3. Mã phân tích dữ liệu

Cell dưới đây chứa **toàn bộ taxonomy, quy tắc lọc, công thức tổng hợp và QA**. Magic `%%writefile` chỉ lưu nguyên code thành script để notebook chạy lại rõ ràng và có thể kiểm tra độc lập.


In [ ]:
%%writefile analyze_seed_market.py
from __future__ import annotations

import argparse
import json
import math
import re
import unicodedata
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd


parser = argparse.ArgumentParser(
    description=(
        "Phân tích thị trường hạt giống từ 3 shop benchmark và dùng "
        "Hiệp Thủy riêng để đo khoảng trống danh mục."
    )
)
parser.add_argument(
    "--input-dir",
    type=Path,
    required=True,
    help="Folder chứa 4 CSV sản phẩm theo đúng tên file mặc định.",
)
parser.add_argument(
    "--output-dir",
    type=Path,
    default=Path("output"),
    help="Folder ghi JSON/CSV trung gian. Mặc định: ./output",
)
parser.add_argument(
    "--analysis-date",
    default=date.today().isoformat(),
    help="Ngày ghi trong workbook, định dạng YYYY-MM-DD.",
)
args = parser.parse_args()

BASE = args.input_dir.expanduser().resolve()
OUTPUT_DIR = args.output_dir.expanduser().resolve()

FILES = {
    "Hà Bắc": BASE / "gia san pham - Ha Bac.csv",
    "Quang Anh VNUA": BASE / "gia san pham - Quang Anh VNUA.csv",
    "Vườn Babylon": BASE / "gia san pham - Vuon Babylon.csv",
    "Hiệp Thủy": BASE / "gia san pham - Hiep Thuy (shop minh).csv",
}
COMPETITORS = ["Hà Bắc", "Quang Anh VNUA", "Vườn Babylon"]
REQUIRED_COLUMNS = {
    "item_id",
    "product_name",
    "product_url",
    "price_min_vnd",
    "price_max_vnd",
    "sold_count",
    "rating_average",
    "rating_count",
    "danh_muc_shop",
    "tinh_trang",
}


def fold_text(value: object) -> str:
    text = str(value or "").lower().replace("đ", "d")
    text = unicodedata.normalize("NFD", text)
    text = "".join(ch for ch in text if unicodedata.category(ch) != "Mn")
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def number(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce").fillna(0)


def is_seed_listing(row: pd.Series) -> bool:
    title = fold_text(row.get("product_name", ""))
    category = fold_text(row.get("danh_muc_shop", ""))
    category_seed = "hat giong" in category
    title_seed = bool(re.search(r"(?:^| )hat giong(?: |$)", title[:95]))
    false_positive = bool(
        re.search(
            r"\b(?:gia the|dat|phan bon|kich re|kich mam|ngam|xu ly|"
            r"thuoc|che pham|qua tang|khay|vi uom|vien nen|xo dua|"
            r"bau uom|chau|dung cu)\b.{0,55}\bhat giong\b|"
            r"\bhat giong\b.{0,45}\b(?:qua tang|kem theo)\b",
            title,
        )
    )
    return bool((category_seed or title_seed) and not false_positive)


def contains(text: str, pattern: str) -> bool:
    return bool(re.search(pattern, text))


def classify_group(title: str) -> str:
    if contains(
        title,
        r"\bcombo\b|\bhon hop\b|\bnhieu loai\b|"
        r"\bhat giong (?:cac loai|rau an la|rau cu qua)\b|"
        r"\bcac loai\b.{0,25}\bhat giong\b",
    ):
        return "Listing tổng hợp / nhiều lựa chọn"
    if contains(title, r"\brau mam\b|\bmam (?:cai|cu|dau|rau)\b"):
        return "Rau mầm"
    if contains(
        title,
        r"\bhat giong hoa\b|\bhoa cuc\b|\bhat giong cuc\b|\bvan tho\b|"
        r"\bhuong duong\b|\bda yen thao\b|"
        r"\bdua can\b|\bmuoi gio\b|\bthach thao\b|\bthuoc duoc\b|"
        r"\bsao nhai\b|\bhoa mao ga\b|\bthien ly\b|\bco la gung\b",
    ):
        return "Hoa & cây cảnh"
    if contains(
        title,
        r"\bngai cuu\b|\bsam duong quy\b|\bduong quy\b|\bco ngot\b|"
        r"\bhuong nhu\b|\bhuong thao\b|\bata?iso\b|\bla giang\b|"
        r"\bcay nhau\b|\bso dua\b",
    ):
        return "Dược liệu & cây đặc sản"
    if contains(
        title,
        r"\bcu cai\b|\bca rot\b|\bsu hao\b|\bcu den\b|\bcu dau\b|"
        r"\bcu san\b|\bkhoai\b|\bcu cai duong\b|\bmang tay\b",
    ):
        return "Rau củ & thân"
    if contains(
        title,
        r"\bdau tay\b|\bdu du\b|"
        r"\bhat giong (?:chanh|cam|buoi|nho|tao|chum ngay|gac|kiwi|thanh long)\b|"
        r"\bcay (?:chanh|cam|buoi|nho|tao|chum ngay|gac|kiwi|thanh long)\b",
    ):
        return "Cây ăn quả & lâu năm"
    if contains(
        title,
        r"\bbap\b|\bngo (?!ri\b|gai\b|bac\b|hat\b|ta\b|mui\b)",
    ):
        return "Bắp / ngô"
    if contains(
        title,
        r"\bdau bap\b|\bdau dua\b|\bdau cove\b|\bdau co ve\b|"
        r"\bdau rong\b|\bdau ha lan\b|\bdau van\b|\bdau xanh\b|"
        r"\bdau den\b|\bdau do\b|\bdau trang\b|\bdau nanh\b|\bdau phong\b",
    ):
        return "Đậu & đậu bắp"
    if contains(
        title,
        r"\bhung\b|\bque\b|\btia to\b|\bkinh gioi\b|"
        r"\bngo (?:ri|gai|bac|hat|ta)\b|\bmui\b|"
        r"\bhanh\b|\bhe\b|\bthi la\b|\bcan tay\b|\bbac ha\b|\bsa\b|"
        r"\brau thom\b|\brau ma\b|\bdiep ca\b|\bbo ngo\b|\be trang\b|"
        r"\bla me\b",
    ):
        return "Rau gia vị & thảo mộc"
    if contains(title, r"\bxa lach\b"):
        return "Xà lách"
    if contains(
        title,
        r"\bcai\b|\bbap cai\b|\bsup lo\b|\bbong cai\b|\bkale\b|"
        r"\bbo xoi\b|\bchan vit\b",
    ):
        return "Cải & rau họ cải"
    if contains(
        title,
        r"\brau muong\b|\brau den\b|\bden\b|\bmong toi\b|\brau day\b|"
        r"\btan o\b|\bcai cuc\b|\brau nhut\b|\brau ngot\b|\brau bo ngot\b|"
        r"\brau cang cua\b|\brau sam\b|\brau diep\b",
    ):
        return "Rau ăn lá khác"
    if contains(title, r"\bdua leo\b|\bdua chuot\b"):
        return "Dưa leo"
    if contains(
        title,
        r"\bdua hau\b|\bdua luoi\b|\bdua le\b|\bdua gang\b|"
        r"\bdua hoang kim\b|\bdua bo\b",
    ):
        return "Dưa lấy quả"
    if contains(title, r"\bca chua\b"):
        return "Cà chua"
    if contains(
        title,
        r"\bbau\b|\bbi\b|\bbi do\b|\bbi dao\b|\bbi ngoi\b|\bbi sat\b|"
        r"\bmuop\b|\bkho qua\b|\bkho hoa\b|\bmuop dang\b|\blac lay\b",
    ):
        return "Bầu, bí, mướp & khổ qua"
    if contains(
        title,
        r"\bot\b|\bca tim\b|\bca phao\b|\bca dia\b|\bca ngot\b|"
        r"\bca bat\b|\bca nau\b|\bca xanh\b|\bca (?!chua\b|rot\b)",
    ):
        return "Cà & ớt"
    return "Khác / cần rà soát"


def classify_cluster(title: str, group: str) -> str:
    if group in {
        "Listing tổng hợp / nhiều lựa chọn",
        "Xà lách",
        "Dưa leo",
        "Bắp / ngô",
        "Cà chua",
    }:
        return group

    if group == "Rau mầm":
        if contains(title, r"\b(?:rau mam|mam).*\b(?:cai|cu cai)\b"):
            return "Rau mầm – cải/củ cải"
        if contains(title, r"\b(?:rau mam|mam).*\b(?:dau|lua|ngo|bap)\b"):
            return "Rau mầm – đậu/ngũ cốc"
        return "Rau mầm – loại khác"

    if group == "Hoa & cây cảnh":
        if contains(title, r"\bhoa cuc\b|\bhat giong cuc\b|\bvan tho\b"):
            return "Hoa – cúc & vạn thọ"
        if contains(title, r"\bhuong duong\b"):
            return "Hoa – hướng dương"
        if contains(title, r"\bda yen thao\b"):
            return "Hoa – dạ yên thảo"
        if contains(title, r"\bdau biec\b"):
            return "Hoa – đậu biếc"
        if contains(title, r"\bco la gung\b|\bco san vuon\b"):
            return "Cỏ sân vườn"
        return "Hoa – loại khác"

    if group == "Dược liệu & cây đặc sản":
        if contains(title, r"\bata?iso\b"):
            return "Atisô"
        if contains(title, r"\bngai cuu\b"):
            return "Ngải cứu"
        if contains(title, r"\bsam duong quy\b|\bduong quy\b"):
            return "Sâm/đương quy"
        return "Dược liệu & cây đặc sản khác"

    if group == "Cây ăn quả & lâu năm":
        if contains(title, r"\bdau tay\b"):
            return "Dâu tây"
        if contains(title, r"\bdu du\b"):
            return "Đu đủ"
        return "Cây ăn quả/lâu năm khác"

    if group == "Rau củ & thân":
        if contains(title, r"\bcu cai\b"):
            return "Củ cải"
        if contains(title, r"\bca rot\b"):
            return "Cà rốt"
        if contains(title, r"\bsu hao\b"):
            return "Su hào"
        if contains(title, r"\bmang tay\b"):
            return "Măng tây"
        return "Rau củ khác"

    if group == "Đậu & đậu bắp":
        if contains(title, r"\bdau bap\b"):
            return "Đậu bắp"
        if contains(title, r"\bdau dua\b|\bdau cove\b|\bdau co ve\b|\bdau van\b"):
            return "Đậu dây/đậu đũa/đậu cove"
        return "Đậu rồng/đậu Hà Lan/đậu khác"

    if group == "Rau gia vị & thảo mộc":
        if contains(title, r"\bhung\b|\bque\b|\btia to\b|\bkinh gioi\b|\bbac ha\b"):
            return "Húng, quế, tía tô & kinh giới"
        if contains(title, r"\bngo (?:ri|gai|bac|hat|ta)\b|\bmui\b"):
            return "Ngò/mùi"
        if contains(title, r"\bhanh\b|\bhe\b"):
            return "Hành & hẹ"
        if contains(title, r"\bthi la\b|\bcan tay\b"):
            return "Thì là & cần tây"
        return "Rau gia vị khác"

    if group == "Cải & rau họ cải":
        if contains(
            title,
            r"\bcai be\b|\bcai ngot\b|\bcai thia\b|\bcai ro\b|"
            r"\bcai mao ga\b|\bcai to sai\b|\bcai tau say\b|"
            r"\bcai xanh\b|\bcai ngong\b",
        ):
            return "Cải bẹ/ngọt/thìa/rổ"
        if contains(title, r"\bbap cai\b|\bcai thao\b"):
            return "Bắp cải & cải thảo"
        if contains(title, r"\bsup lo\b|\bbong cai\b"):
            return "Súp lơ/bông cải"
        if contains(
            title,
            r"\bkale\b|\bbo xoi\b|\bchan vit\b|\bcai meo\b|"
            r"\bcai cau vong\b|\bcai hoa hong\b",
        ):
            return "Kale, bó xôi & cải đặc sản"
        return "Cải khác"

    if group == "Rau ăn lá khác":
        if contains(title, r"\brau muong\b"):
            return "Rau muống"
        if contains(title, r"\brau den\b|\bden (?:do|xanh|tieu|tia)\b"):
            return "Rau dền"
        return "Mồng tơi, rau đay & rau lá khác"

    if group == "Dưa lấy quả":
        if contains(title, r"\bdua hau\b"):
            return "Dưa hấu"
        if contains(title, r"\bdua luoi\b|\bdua le\b|\bdua gang\b|\bdua hoang kim\b"):
            return "Dưa lưới/lê/gang"
        return "Dưa lấy quả khác"

    if group == "Bầu, bí, mướp & khổ qua":
        if contains(title, r"\bbau\b|\blac lay\b"):
            return "Bầu"
        if contains(title, r"\bbi\b|\bbi do\b|\bbi dao\b|\bbi ngoi\b|\bbi sat\b"):
            return "Bí"
        if contains(title, r"\bmuop\b"):
            return "Mướp"
        return "Khổ qua/mướp đắng"

    if group == "Cà & ớt":
        if contains(title, r"\bot\b"):
            return "Ớt"
        return "Cà tím/cà pháo/cà khác"

    return group


def percentile_index(values: pd.Series) -> pd.Series:
    if len(values) == 1:
        return pd.Series([100.0], index=values.index)
    return values.rank(method="average", pct=True) * 100


def aggregate(frame: pd.DataFrame, key: str) -> pd.DataFrame:
    valid_price = frame[frame["price_mid_vnd"] > 0]
    base = (
        frame.groupby(key, dropna=False)
        .agg(
            product_count=("item_id", "nunique"),
            shop_coverage=("shop_name", "nunique"),
            sold_count=("sold_count", "sum"),
            review_count=("rating_count", "sum"),
        )
        .reset_index()
    )
    price = (
        valid_price.groupby(key)["price_mid_vnd"]
        .agg(
            price_p25_vnd=lambda x: float(x.quantile(0.25)),
            price_median_vnd="median",
            price_p75_vnd=lambda x: float(x.quantile(0.75)),
            price_min_observed_vnd="min",
            price_max_observed_vnd="max",
        )
        .reset_index()
    )
    concentration_rows = []
    for label, group in frame.groupby(key, dropna=False):
        sold = group["sold_count"].sort_values(ascending=False)
        total = float(sold.sum())
        concentration_rows.append(
            {
                key: label,
                "top1_sold_share": float(sold.iloc[0] / total) if total else 0.0,
                "top3_sold_share": float(sold.head(3).sum() / total) if total else 0.0,
            }
        )
    concentration = pd.DataFrame(concentration_rows)
    out = base.merge(price, on=key, how="left").merge(
        concentration, on=key, how="left"
    )
    out["sold_index"] = percentile_index(out["sold_count"])
    out["review_index"] = percentile_index(out["review_count"])
    out["demand_index"] = (out["sold_index"] + out["review_index"]) / 2
    out["demand_tier"] = pd.cut(
        out["demand_index"],
        bins=[-np.inf, 50, 75, np.inf],
        labels=["Thấp", "Trung bình", "Cao"],
    ).astype(str)
    return out.sort_values(
        ["demand_index", "sold_count", "review_count"],
        ascending=[False, False, False],
    ).reset_index(drop=True)


def round_numeric(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    for column in out.select_dtypes(include=["float"]).columns:
        digits = 4 if column.endswith("_share") else 1
        out[column] = out[column].round(digits)
    return out


frames = []
for shop_name, path in FILES.items():
    if not path.exists():
        raise FileNotFoundError(f"Không tìm thấy file cho {shop_name}: {path}")
    frame = pd.read_csv(path, dtype=str, encoding="utf-8-sig").fillna("")
    missing_columns = sorted(REQUIRED_COLUMNS - set(frame.columns))
    if missing_columns:
        raise ValueError(
            f"{path.name} thiếu cột bắt buộc: {', '.join(missing_columns)}"
        )
    frame["shop_name"] = shop_name
    frame["source_file"] = path.name
    frames.append(frame)

raw = pd.concat(frames, ignore_index=True)
raw["is_seed_listing"] = raw.apply(is_seed_listing, axis=1)
seed = raw[raw["is_seed_listing"]].copy()
seed["title_fold"] = seed["product_name"].map(fold_text)
seed["sold_count"] = number(seed["sold_count"])
seed["rating_count"] = number(seed["rating_count"])
seed["price_min_vnd"] = number(seed["price_min_vnd"])
seed["price_max_vnd"] = number(seed["price_max_vnd"])
seed["price_mid_vnd"] = np.where(
    (seed["price_min_vnd"] > 0) & (seed["price_max_vnd"] > 0),
    (seed["price_min_vnd"] + seed["price_max_vnd"]) / 2,
    np.maximum(seed["price_min_vnd"], seed["price_max_vnd"]),
)
seed["market_group"] = seed["title_fold"].map(classify_group)
seed["product_cluster"] = [
    classify_cluster(title, group)
    for title, group in zip(seed["title_fold"], seed["market_group"])
]

competitor_seed = seed[seed["shop_name"].isin(COMPETITORS)].copy()
hiep_thuy_seed = seed[seed["shop_name"].eq("Hiệp Thủy")].copy()

group_summary = aggregate(competitor_seed, "market_group")
group_summary.insert(0, "demand_rank", range(1, len(group_summary) + 1))

cluster_summary = aggregate(competitor_seed, "product_cluster")
ht_cluster = (
    hiep_thuy_seed.groupby("product_cluster")
    .agg(
        ht_product_count=("item_id", "nunique"),
        ht_sold_count=("sold_count", "sum"),
        ht_review_count=("rating_count", "sum"),
    )
    .reset_index()
)
cluster_gap = cluster_summary.merge(ht_cluster, on="product_cluster", how="left")
for column in ["ht_product_count", "ht_sold_count", "ht_review_count"]:
    cluster_gap[column] = cluster_gap[column].fillna(0)
competitor_average = cluster_gap["product_count"] / len(COMPETITORS)
cluster_gap["portfolio_gap_score"] = (
    100 * (1 - np.minimum(cluster_gap["ht_product_count"] / competitor_average, 1))
).replace([np.inf, -np.inf], 0)
cluster_gap["coverage_score"] = 100 * cluster_gap["shop_coverage"] / len(COMPETITORS)
cluster_gap["evidence_breadth_score"] = (
    50 * np.minimum(cluster_gap["product_count"] / 10, 1)
    + 50 * (1 - cluster_gap["top1_sold_share"])
)
cluster_gap["opportunity_score"] = (
    0.50 * cluster_gap["demand_index"]
    + 0.15 * cluster_gap["coverage_score"]
    + 0.20 * cluster_gap["portfolio_gap_score"]
    + 0.15 * cluster_gap["evidence_breadth_score"]
)
cluster_gap["ht_status"] = np.select(
    [
        cluster_gap["ht_product_count"].eq(0),
        cluster_gap["portfolio_gap_score"].ge(50),
    ],
    ["Chưa có", "Có nhưng còn mỏng"],
    default="Đã phủ tương đối",
)
cluster_gap["recommended_action"] = np.select(
    [
        cluster_gap["ht_status"].eq("Chưa có")
        & cluster_gap["demand_index"].ge(60),
        cluster_gap["ht_status"].eq("Chưa có"),
        cluster_gap["ht_status"].eq("Có nhưng còn mỏng")
        & cluster_gap["demand_index"].ge(60),
        cluster_gap["ht_status"].eq("Có nhưng còn mỏng"),
    ],
    [
        "Thử nghiệm 3–5 SKU",
        "Theo dõi / test nhỏ",
        "Mở rộng thêm biến thể",
        "Bổ sung chọn lọc",
    ],
    default="Tối ưu SKU hiện có",
)
cluster_gap = cluster_gap.sort_values(
    ["opportunity_score", "demand_index", "sold_count"],
    ascending=[False, False, False],
).reset_index(drop=True)
cluster_gap.insert(0, "opportunity_rank", range(1, len(cluster_gap) + 1))

missing_clusters = cluster_gap[cluster_gap["ht_product_count"].eq(0)].copy()
potential = cluster_gap[
    cluster_gap["ht_status"].isin(["Chưa có", "Có nhưng còn mỏng"])
].head(12)

qa = {
    "source_rows_by_shop": raw.groupby("shop_name").size().astype(int).to_dict(),
    "seed_products_by_shop": seed.groupby("shop_name")["item_id"].nunique().astype(int).to_dict(),
    "competitor_seed_products": int(competitor_seed["item_id"].nunique()),
    "hiep_thuy_seed_products": int(hiep_thuy_seed["item_id"].nunique()),
    "unclassified_competitor_products": int(
        competitor_seed["market_group"].eq("Khác / cần rà soát").sum()
    ),
    "unclassified_ht_products": int(
        hiep_thuy_seed["market_group"].eq("Khác / cần rà soát").sum()
    ),
    "missing_price_competitor_products": int(
        competitor_seed["price_mid_vnd"].le(0).sum()
    ),
    "duplicate_shop_item": int(seed.duplicated(["shop_name", "item_id"]).sum()),
}

selected_columns = [
    "shop_name",
    "item_id",
    "product_name",
    "market_group",
    "product_cluster",
    "price_min_vnd",
    "price_max_vnd",
    "price_mid_vnd",
    "sold_count",
    "rating_count",
    "rating_average",
    "danh_muc_shop",
    "tinh_trang",
    "product_url",
    "source_file",
]
source_seed = seed[selected_columns].copy()

payload = {
    "analysis_date": args.analysis_date,
    "input_files": {shop: str(path.resolve()) for shop, path in FILES.items()},
    "qa": qa,
    "competitors": COMPETITORS,
    "group_summary": round_numeric(group_summary).to_dict(orient="records"),
    "cluster_gap": round_numeric(cluster_gap).to_dict(orient="records"),
    "missing_clusters": round_numeric(missing_clusters).to_dict(orient="records"),
    "potential": round_numeric(potential).to_dict(orient="records"),
    "source_seed": round_numeric(source_seed).to_dict(orient="records"),
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "seed_market_analysis.json").write_text(
    json.dumps(payload, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
round_numeric(group_summary).to_csv(
    OUTPUT_DIR / "group_summary.csv", index=False, encoding="utf-8-sig"
)
round_numeric(cluster_gap).to_csv(
    OUTPUT_DIR / "cluster_gap.csv", index=False, encoding="utf-8-sig"
)
round_numeric(source_seed).to_csv(
    OUTPUT_DIR / "source_seed_products.csv", index=False, encoding="utf-8-sig"
)

print(json.dumps(qa, ensure_ascii=False, indent=2))
print("\nTOP GROUPS")
print(
    group_summary[
        [
            "demand_rank",
            "market_group",
            "product_count",
            "shop_coverage",
            "sold_count",
            "review_count",
            "price_p25_vnd",
            "price_median_vnd",
            "price_p75_vnd",
            "demand_index",
        ]
    ].head(15).to_string(index=False)
)
print("\nTOP OPPORTUNITIES")
print(
    cluster_gap[
        [
            "opportunity_rank",
            "product_cluster",
            "product_count",
            "shop_coverage",
            "sold_count",
            "review_count",
            "demand_index",
            "ht_product_count",
            "ht_status",
            "opportunity_score",
        ]
    ].head(20).to_string(index=False)
)


## 4. Chạy phân tích và xem kết quả chính

In [ ]:
analysis_command = [
    sys.executable,
    str(Path.cwd() / "analyze_seed_market.py"),
    "--input-dir", str(INPUT_DIR),
    "--output-dir", str(OUTPUT_DIR),
    "--analysis-date", ANALYSIS_DATE,
]
analysis_run = subprocess.run(analysis_command, check=True, text=True, capture_output=True)
print(analysis_run.stdout)

ANALYSIS_JSON = OUTPUT_DIR / "seed_market_analysis.json"
analysis_payload = json.loads(ANALYSIS_JSON.read_text(encoding="utf-8"))
group_summary = pd.DataFrame(analysis_payload["group_summary"])
cluster_gap = pd.DataFrame(analysis_payload["cluster_gap"])
source_seed = pd.DataFrame(analysis_payload["source_seed"])

print("Top 8 nhóm theo nhu cầu")
display(group_summary[[
    "demand_rank", "market_group", "product_count", "shop_coverage",
    "price_p25_vnd", "price_median_vnd", "price_p75_vnd",
    "sold_count", "review_count", "demand_index", "demand_tier",
]].head(8))

print("Top 10 cơ hội danh mục")
display(cluster_gap[[
    "opportunity_rank", "product_cluster", "product_count", "shop_coverage",
    "sold_count", "review_count", "ht_product_count", "ht_status", "opportunity_score",
]].head(10))


## 5. Kiểm tra dữ liệu và phương pháp

In [ ]:
expected_snapshot = {
    "competitor_seed_products": 680,
    "hiep_thuy_seed_products": 111,
    "unclassified_competitor_products": 0,
    "unclassified_ht_products": 0,
    "missing_price_competitor_products": 0,
    "duplicate_shop_item": 0,
}

qa_rows = []
for metric, expected in expected_snapshot.items():
    actual = analysis_payload["qa"].get(metric)
    qa_rows.append({"check": metric, "expected": expected, "actual": actual, "pass": actual == expected})
qa_rows.extend(
    [
        {"check": "market_groups", "expected": 17, "actual": len(group_summary), "pass": len(group_summary) == 17},
        {"check": "product_clusters", "expected": 50, "actual": len(cluster_gap), "pass": len(cluster_gap) == 50},
        {"check": "classified_seed_skus", "expected": 791, "actual": len(source_seed), "pass": len(source_seed) == 791},
        {"check": "benchmark_shop_set", "expected": "3 shop đã chốt", "actual": ", ".join(analysis_payload["competitors"]), "pass": analysis_payload["competitors"] == ["Hà Bắc", "Quang Anh VNUA", "Vườn Babylon"]},
        {"check": "An Việt excluded", "expected": 0, "actual": int((source_seed["shop_name"] == "An Việt").sum()), "pass": int((source_seed["shop_name"] == "An Việt").sum()) == 0},
    ]
)
qa_table = pd.DataFrame(qa_rows)
display(qa_table)
assert qa_table["pass"].all(), "DATA QA FAILED — xem bảng kiểm tra phía trên"
print("FINAL DATA QA: PASS")


## 6. Mã tạo file Excel

Phần xuất workbook dùng `XlsxWriter` để notebook chạy độc lập trên Google Colab. Workbook vẫn giữ năm sheet, công thức Excel, bảng, định dạng có điều kiện và biểu đồ.


In [ ]:
%%writefile build_seed_market_workbook_colab.py
from __future__ import annotations

import argparse
import json
import math
from pathlib import Path

import xlsxwriter
from xlsxwriter.utility import xl_range


COLORS = {
    "green": "#1F4D3A",
    "green2": "#2F6B4F",
    "green_light": "#E8F3ED",
    "amber_light": "#FFF1D6",
    "red": "#B94A48",
    "red_light": "#FBE4E3",
    "blue_light": "#E6F0FA",
    "ink": "#1F2937",
    "gray": "#6B7280",
    "gray_light": "#F3F4F6",
    "line": "#D1D5DB",
    "white": "#FFFFFF",
}


def js_round(value: object) -> int:
    number = float(value or 0)
    return math.floor(number + 0.5) if number >= 0 else math.ceil(number - 0.5)


def money(value: object) -> str:
    return f"{js_round(value):,} ₫"


def price_band(record: dict) -> str:
    return f"{money(record.get('price_p25_vnd'))} – {money(record.get('price_p75_vnd'))}"


def literal_safe(value: object) -> object:
    if value is None:
        return ""
    if isinstance(value, str) and value.startswith(("=", "+", "-", "@")):
        return "'" + value
    return value


def as_number(value: object, default: float = 0) -> float:
    if value in (None, ""):
        return default
    try:
        return float(value)
    except (TypeError, ValueError):
        return default


def build_formats(workbook: xlsxwriter.Workbook) -> dict:
    return {
        "title": workbook.add_format(
            {
                "bg_color": COLORS["green"],
                "font_color": COLORS["white"],
                "bold": True,
                "font_size": 18,
                "align": "left",
                "valign": "vcenter",
            }
        ),
        "note_green": workbook.add_format(
            {
                "bg_color": COLORS["green_light"],
                "font_color": COLORS["ink"],
                "italic": True,
                "text_wrap": True,
                "valign": "vcenter",
            }
        ),
        "note_amber": workbook.add_format(
            {
                "bg_color": COLORS["amber_light"],
                "font_color": "#7A4A00",
                "italic": True,
                "text_wrap": True,
                "valign": "vcenter",
            }
        ),
        "section": workbook.add_format(
            {
                "bg_color": COLORS["green2"],
                "font_color": COLORS["white"],
                "bold": True,
                "font_size": 11,
                "valign": "vcenter",
            }
        ),
        "header": workbook.add_format(
            {
                "bg_color": COLORS["green2"],
                "font_color": COLORS["white"],
                "bold": True,
                "align": "center",
                "valign": "vcenter",
                "text_wrap": True,
                "border": 1,
                "border_color": COLORS["line"],
            }
        ),
        "money": workbook.add_format({"num_format": '#,##0 "₫"'}),
        "integer": workbook.add_format({"num_format": "#,##0"}),
        "decimal": workbook.add_format({"num_format": "0.0"}),
        "percent": workbook.add_format({"num_format": "0.0%"}),
        "wrap": workbook.add_format({"text_wrap": True, "valign": "top"}),
        "body_border": workbook.add_format(
            {
                "border": 1,
                "border_color": COLORS["line"],
                "text_wrap": True,
                "valign": "top",
            }
        ),
        "priority": workbook.add_format(
            {
                "bg_color": COLORS["amber_light"],
                "font_color": "#8A5A00",
                "bold": True,
                "align": "center",
                "valign": "vcenter",
                "border": 1,
                "border_color": COLORS["line"],
            }
        ),
        "kpi_label": workbook.add_format(
            {
                "bg_color": COLORS["green2"],
                "font_color": COLORS["white"],
                "bold": True,
                "align": "center",
                "valign": "vcenter",
                "text_wrap": True,
            }
        ),
        "kpi_value": workbook.add_format(
            {
                "bg_color": COLORS["green_light"],
                "font_color": COLORS["green"],
                "bold": True,
                "font_size": 20,
                "align": "center",
                "valign": "vcenter",
                "num_format": "#,##0",
                "border": 1,
                "border_color": COLORS["line"],
            }
        ),
        "helper": workbook.add_format(
            {
                "bg_color": COLORS["gray_light"],
                "font_color": COLORS["gray"],
                "font_size": 9,
            }
        ),
        "blue_note": workbook.add_format(
            {
                "bg_color": COLORS["blue_light"],
                "font_color": COLORS["ink"],
                "italic": True,
                "text_wrap": True,
                "valign": "vcenter",
            }
        ),
        "method_label": workbook.add_format(
            {
                "bg_color": COLORS["green_light"],
                "font_color": COLORS["green"],
                "bold": True,
                "valign": "top",
                "border": 1,
                "border_color": COLORS["line"],
            }
        ),
        "method_body": workbook.add_format(
            {
                "text_wrap": True,
                "valign": "top",
                "border": 1,
                "border_color": COLORS["line"],
            }
        ),
        "italic_note": workbook.add_format(
            {"italic": True, "font_color": COLORS["ink"], "text_wrap": True}
        ),
    }


def write_title(sheet, last_col: int, title: str, fmt: dict) -> None:
    sheet.merge_range(0, 0, 0, last_col, title, fmt["title"])
    sheet.set_row(0, 34)


def add_table(sheet, first_row: int, first_col: int, last_row: int, last_col: int, name: str, headers: list[str]) -> None:
    sheet.add_table(
        first_row,
        first_col,
        last_row,
        last_col,
        {
            "name": name,
            "style": "Table Style Medium 4",
            "banded_rows": True,
            "autofilter": True,
            "columns": [{"header": header} for header in headers],
        },
    )


def build_source_sheet(workbook, sheet, data: dict, fmt: dict) -> dict:
    write_title(sheet, 11, "SKU HẠT GIỐNG ĐÃ PHÂN LOẠI — 3 ĐỐI THỦ + HIỆP THỦY", fmt)
    sheet.merge_range(1, 0, 1, 11, "Giá đại diện là midpoint của price_min_vnd và price_max_vnd.", fmt["note_green"])
    headers = [
        "Shop", "Item ID", "Tên sản phẩm", "Nhóm thị trường", "Cụm sản phẩm",
        "Giá min (VND)", "Giá max (VND)", "Giá đại diện (VND)",
        "Sold count", "Số review", "Rating trung bình", "Danh mục shop",
    ]
    header_row = 2
    data_start = 3
    for col, header in enumerate(headers):
        sheet.write(header_row, col, header, fmt["header"])
    for index, record in enumerate(data["source_seed"]):
        row = data_start + index
        price_mid = as_number(record.get("price_mid_vnd"))
        values = [
            literal_safe(record.get("shop_name")),
            str(record.get("item_id") or ""),
            literal_safe(record.get("product_name")),
            literal_safe(record.get("market_group")),
            literal_safe(record.get("product_cluster")),
            as_number(record.get("price_min_vnd")),
            as_number(record.get("price_max_vnd")),
            None,
            as_number(record.get("sold_count")),
            as_number(record.get("rating_count")),
            None if record.get("rating_average") in (None, "") else as_number(record.get("rating_average")),
            literal_safe(record.get("danh_muc_shop")),
        ]
        for col, value in enumerate(values):
            cell_format = None
            if 5 <= col <= 7:
                cell_format = fmt["money"]
            elif col in (8, 9):
                cell_format = fmt["integer"]
            elif col == 10:
                cell_format = fmt["decimal"]
            elif col in (2, 3, 4, 11):
                cell_format = fmt["wrap"]
            if col == 7:
                excel_row = row + 1
                formula = f"=IF(AND(F{excel_row}>0,G{excel_row}>0),(F{excel_row}+G{excel_row})/2,MAX(F{excel_row},G{excel_row}))"
                sheet.write_formula(row, col, formula, cell_format, price_mid)
            elif value is None:
                sheet.write_blank(row, col, None, cell_format)
            else:
                sheet.write(row, col, value, cell_format)
    end_row = data_start + len(data["source_seed"]) - 1
    add_table(sheet, header_row, 0, end_row, 11, "SeedProductSourceTable", headers)
    sheet.freeze_panes(3, 2)
    sheet.hide_gridlines(2)
    widths = [18, 15, 52, 26, 26, 16, 16, 16, 16, 16, 16, 24]
    for col, width in enumerate(widths):
        sheet.set_column(col, col, width)
    sheet.set_row(header_row, 32)
    return {"data_start_excel": data_start + 1, "data_end_excel": end_row + 1}


def build_group_sheet(workbook, sheet, data: dict, fmt: dict, source_bounds: dict) -> dict:
    write_title(sheet, 14, "CÁC NHÓM HẠT GIỐNG PHỔ BIẾN VÀ KHOẢNG GIÁ", fmt)
    sheet.merge_range(
        1, 0, 1, 14,
        "Benchmark: Hà Bắc, Quang Anh VNUA và Vườn Babylon. Khoảng giá phổ biến = P25–P75 của giá đại diện/listing.",
        fmt["note_green"],
    )
    headers = [
        "Hạng nhu cầu", "Nhóm thị trường", "Số listing đối thủ", "Số shop có nhóm",
        "Giá P25 (VND)", "Giá median (VND)", "Giá P75 (VND)", "Sold count",
        "Số review", "Chỉ số sold", "Chỉ số review", "Chỉ số nhu cầu",
        "Mức nhu cầu", "Top 1 sold share", "Top 3 sold share",
    ]
    header_row = 2
    data_start = 3
    src_start = source_bounds["data_start_excel"]
    src_end = source_bounds["data_end_excel"]
    for col, header in enumerate(headers):
        sheet.write(header_row, col, header, fmt["header"])
    for index, record in enumerate(data["group_summary"]):
        row = data_start + index
        excel_row = row + 1
        exact_demand_index = (
            as_number(record.get("sold_index"))
            + as_number(record.get("review_index"))
        ) / 2
        static_values = {
            0: as_number(record.get("demand_rank")),
            1: record.get("market_group", ""),
            4: as_number(record.get("price_p25_vnd")),
            5: as_number(record.get("price_median_vnd")),
            6: as_number(record.get("price_p75_vnd")),
            9: as_number(record.get("sold_index")),
            10: as_number(record.get("review_index")),
            13: as_number(record.get("top1_sold_share")),
            14: as_number(record.get("top3_sold_share")),
        }
        for col, value in static_values.items():
            cell_fmt = fmt["money"] if col in (4, 5, 6) else fmt["percent"] if col in (13, 14) else fmt["decimal"] if col in (9, 10) else fmt["integer"] if col == 0 else None
            sheet.write(row, col, value, cell_fmt)
        group_ref = f"B{excel_row}"
        formulas = {
            2: (
                f'=COUNTIFS(\'SKU đã phân loại\'!$D${src_start}:$D${src_end},{group_ref},'
                f'\'SKU đã phân loại\'!$A${src_start}:$A${src_end},"<>Hiệp Thủy")',
                as_number(record.get("product_count")),
                fmt["integer"],
            ),
            3: (
                f'=--(COUNTIFS(\'SKU đã phân loại\'!$D${src_start}:$D${src_end},{group_ref},\'SKU đã phân loại\'!$A${src_start}:$A${src_end},"Hà Bắc")>0)'
                f'+--(COUNTIFS(\'SKU đã phân loại\'!$D${src_start}:$D${src_end},{group_ref},\'SKU đã phân loại\'!$A${src_start}:$A${src_end},"Quang Anh VNUA")>0)'
                f'+--(COUNTIFS(\'SKU đã phân loại\'!$D${src_start}:$D${src_end},{group_ref},\'SKU đã phân loại\'!$A${src_start}:$A${src_end},"Vườn Babylon")>0)',
                as_number(record.get("shop_coverage")),
                fmt["integer"],
            ),
            7: (
                f'=SUMIFS(\'SKU đã phân loại\'!$I${src_start}:$I${src_end},\'SKU đã phân loại\'!$D${src_start}:$D${src_end},{group_ref},\'SKU đã phân loại\'!$A${src_start}:$A${src_end},"<>Hiệp Thủy")',
                as_number(record.get("sold_count")),
                fmt["integer"],
            ),
            8: (
                f'=SUMIFS(\'SKU đã phân loại\'!$J${src_start}:$J${src_end},\'SKU đã phân loại\'!$D${src_start}:$D${src_end},{group_ref},\'SKU đã phân loại\'!$A${src_start}:$A${src_end},"<>Hiệp Thủy")',
                as_number(record.get("review_count")),
                fmt["integer"],
            ),
            11: (f"=(J{excel_row}+K{excel_row})/2", exact_demand_index, fmt["decimal"]),
            12: (f'=IF(L{excel_row}>75,"Cao",IF(L{excel_row}>50,"Trung bình","Thấp"))', record.get("demand_tier", ""), None),
        }
        for col, (formula, cached, cell_fmt) in formulas.items():
            sheet.write_formula(row, col, formula, cell_fmt, cached)
    end_row = data_start + len(data["group_summary"]) - 1
    add_table(sheet, header_row, 0, end_row, 14, "MarketGroupTable", headers)
    sheet.conditional_format(data_start, 12, end_row, 12, {"type": "text", "criteria": "containing", "value": "Cao", "format": workbook.add_format({"bg_color": COLORS["green_light"], "font_color": COLORS["green"], "bold": True})})
    sheet.conditional_format(data_start, 12, end_row, 12, {"type": "text", "criteria": "containing", "value": "Trung bình", "format": workbook.add_format({"bg_color": COLORS["amber_light"], "font_color": "#8A5A00"})})
    sheet.conditional_format(data_start, 12, end_row, 12, {"type": "text", "criteria": "containing", "value": "Thấp", "format": workbook.add_format({"bg_color": COLORS["gray_light"], "font_color": COLORS["gray"]})})
    sheet.freeze_panes(3, 0)
    sheet.hide_gridlines(2)
    widths = [14, 34, 16, 16, 17, 17, 17, 15, 15, 15, 15, 15, 16, 18, 18]
    for col, width in enumerate(widths):
        sheet.set_column(col, col, width)
    sheet.set_row(header_row, 32)
    notes = {
        21: "P25: mức giá mà 25% sản phẩm có giá thấp hơn hoặc bằng.",
        22: "P75: mức giá mà 75% sản phẩm có giá thấp hơn hoặc bằng.",
        23: "P25–P75: khoảng chứa 50% sản phẩm nằm giữa thị trường, giúp loại bớt các mức giá quá thấp hoặc quá cao.",
        25: "Ví dụ nhóm cà chua có P25–P75 là 12.960–20.000đ, nghĩa là khoảng một nửa số listing cà chua có giá trong dải này. Đây được xem là “khoảng giá phổ biến",
    }
    for row, text in notes.items():
        sheet.merge_range(row, 0, row, 14, text, fmt["italic_note"])
        sheet.set_row(row, 34 if row == 25 else 22)
    return {"data_start_excel": data_start + 1, "data_end_excel": end_row + 1}


def build_gap_sheet(workbook, sheet, data: dict, fmt: dict, source_bounds: dict) -> dict:
    write_title(sheet, 18, "KHOẢNG TRỐNG DANH MỤC VÀ CƠ HỘI CHO HIỆP THỦY", fmt)
    sheet.merge_range(
        1, 0, 1, 18,
        "Điểm cơ hội = 50% nhu cầu + 15% độ phủ shop + 20% khoảng trống danh mục + 15% độ rộng bằng chứng. Xem cột Top 1 share để tránh kết luận từ một listing đơn lẻ.",
        fmt["note_green"],
    )
    headers = [
        "Hạng cơ hội", "Cụm sản phẩm", "Listing đối thủ", "Shop coverage", "Sold count",
        "Số review", "Giá P25", "Giá median", "Giá P75", "Listing HT", "Portfolio gap",
        "Trạng thái HT", "Chỉ số nhu cầu", "Top 1 sold share", "Độ rộng bằng chứng",
        "Điểm cơ hội", "Ví dụ dẫn đầu", "Shop ví dụ", "Sold ví dụ",
    ]
    top_examples: dict[str, dict] = {}
    for record in data["source_seed"]:
        if record.get("shop_name") == "Hiệp Thủy":
            continue
        cluster = record.get("product_cluster", "")
        current = top_examples.get(cluster)
        if current is None or as_number(record.get("sold_count")) > as_number(current.get("sold_count")):
            top_examples[cluster] = record
    header_row = 2
    data_start = 3
    src_start = source_bounds["data_start_excel"]
    src_end = source_bounds["data_end_excel"]
    for col, header in enumerate(headers):
        sheet.write(header_row, col, header, fmt["header"])
    for index, record in enumerate(data["cluster_gap"]):
        row = data_start + index
        excel_row = row + 1
        example = top_examples.get(record.get("product_cluster", ""), {})
        product_count = as_number(record.get("product_count"))
        shop_coverage = as_number(record.get("shop_coverage"))
        ht_product_count = as_number(record.get("ht_product_count"))
        top1_sold_share = as_number(record.get("top1_sold_share"))
        portfolio_gap = (
            0
            if product_count == 0
            else 100 * (1 - min(ht_product_count / (product_count / 3), 1))
        )
        evidence_breadth = 50 * min(product_count / 10, 1) + 50 * (1 - top1_sold_share)
        opportunity_score = (
            0.5 * as_number(record.get("demand_index"))
            + 0.15 * (100 * shop_coverage / 3)
            + 0.2 * portfolio_gap
            + 0.15 * evidence_breadth
        )
        static_values = {
            0: as_number(record.get("opportunity_rank")),
            1: record.get("product_cluster", ""),
            6: as_number(record.get("price_p25_vnd")),
            7: as_number(record.get("price_median_vnd")),
            8: as_number(record.get("price_p75_vnd")),
            12: as_number(record.get("demand_index")),
            13: as_number(record.get("top1_sold_share")),
            16: literal_safe(example.get("product_name", "")),
            17: example.get("shop_name", ""),
            18: as_number(example.get("sold_count")),
        }
        for col, value in static_values.items():
            cell_fmt = fmt["money"] if col in (6, 7, 8) else fmt["percent"] if col == 13 else fmt["integer"] if col in (0, 18) else fmt["decimal"] if col == 12 else fmt["wrap"] if col in (1, 16, 17) else None
            sheet.write(row, col, value, cell_fmt)
        cluster_ref = f"B{excel_row}"
        formulas = {
            2: (f'=COUNTIFS(\'SKU đã phân loại\'!$E${src_start}:$E${src_end},{cluster_ref},\'SKU đã phân loại\'!$A${src_start}:$A${src_end},"<>Hiệp Thủy")', as_number(record.get("product_count")), fmt["integer"]),
            3: (f'=--(COUNTIFS(\'SKU đã phân loại\'!$E${src_start}:$E${src_end},{cluster_ref},\'SKU đã phân loại\'!$A${src_start}:$A${src_end},"Hà Bắc")>0)+--(COUNTIFS(\'SKU đã phân loại\'!$E${src_start}:$E${src_end},{cluster_ref},\'SKU đã phân loại\'!$A${src_start}:$A${src_end},"Quang Anh VNUA")>0)+--(COUNTIFS(\'SKU đã phân loại\'!$E${src_start}:$E${src_end},{cluster_ref},\'SKU đã phân loại\'!$A${src_start}:$A${src_end},"Vườn Babylon")>0)', as_number(record.get("shop_coverage")), fmt["integer"]),
            4: (f'=SUMIFS(\'SKU đã phân loại\'!$I${src_start}:$I${src_end},\'SKU đã phân loại\'!$E${src_start}:$E${src_end},{cluster_ref},\'SKU đã phân loại\'!$A${src_start}:$A${src_end},"<>Hiệp Thủy")', as_number(record.get("sold_count")), fmt["integer"]),
            5: (f'=SUMIFS(\'SKU đã phân loại\'!$J${src_start}:$J${src_end},\'SKU đã phân loại\'!$E${src_start}:$E${src_end},{cluster_ref},\'SKU đã phân loại\'!$A${src_start}:$A${src_end},"<>Hiệp Thủy")', as_number(record.get("review_count")), fmt["integer"]),
            9: (f'=COUNTIFS(\'SKU đã phân loại\'!$E${src_start}:$E${src_end},{cluster_ref},\'SKU đã phân loại\'!$A${src_start}:$A${src_end},"Hiệp Thủy")', as_number(record.get("ht_product_count")), fmt["integer"]),
            10: (f"=IF(C{excel_row}=0,0,100*(1-MIN(J{excel_row}/(C{excel_row}/3),1)))", portfolio_gap, fmt["decimal"]),
            11: (f'=IF(J{excel_row}=0,"Chưa có",IF(K{excel_row}>=50,"Có nhưng còn mỏng","Đã phủ tương đối"))', record.get("ht_status", ""), None),
            14: (f"=50*MIN(C{excel_row}/10,1)+50*(1-N{excel_row})", evidence_breadth, fmt["decimal"]),
            15: (f"=0.5*M{excel_row}+0.15*(100*D{excel_row}/3)+0.2*K{excel_row}+0.15*O{excel_row}", opportunity_score, fmt["decimal"]),
        }
        for col, (formula, cached, cell_fmt) in formulas.items():
            sheet.write_formula(row, col, formula, cell_fmt, cached)
    end_row = data_start + len(data["cluster_gap"]) - 1
    add_table(sheet, header_row, 0, end_row, 18, "GapOpportunityTable", headers)
    status_formats = {
        "Chưa có": workbook.add_format({"bg_color": COLORS["red_light"], "font_color": COLORS["red"], "bold": True}),
        "Có nhưng còn mỏng": workbook.add_format({"bg_color": COLORS["amber_light"], "font_color": "#8A5A00"}),
        "Đã phủ tương đối": workbook.add_format({"bg_color": COLORS["green_light"], "font_color": COLORS["green"]}),
    }
    for text, cell_format in status_formats.items():
        sheet.conditional_format(data_start, 11, end_row, 11, {"type": "text", "criteria": "containing", "value": text, "format": cell_format})
    sheet.conditional_format(data_start, 15, end_row, 15, {"type": "data_bar", "bar_color": COLORS["green2"], "bar_solid": False})
    sheet.freeze_panes(3, 2)
    sheet.hide_gridlines(2)
    widths = [13, 34, 14, 14, 14, 14, 15, 15, 15, 16, 16, 16, 16, 16, 16, 16, 56, 16, 16]
    for col, width in enumerate(widths):
        sheet.set_column(col, col, width)
    sheet.set_row(header_row, 32)
    return {"data_start_excel": data_start + 1, "data_end_excel": end_row + 1}


def build_summary_sheet(workbook, sheet, data: dict, fmt: dict, group_bounds: dict, gap_bounds: dict, source_bounds: dict) -> None:
    write_title(sheet, 15, "HẠT GIỐNG TRÊN SHOPEE — NHU CẦU, GIÁ VÀ CƠ HỘI CHO HIỆP THỦY", fmt)
    sheet.merge_range(
        1, 0, 1, 15,
        "Phạm vi benchmark chỉ gồm Hà Bắc, Quang Anh VNUA và Vườn Babylon; không tính An Việt hoặc Hiệp Thủy. Hiệp Thủy chỉ dùng để đo gap. Sold count/review là số tích lũy.",
        fmt["note_amber"],
    )
    group_start, group_end = group_bounds["data_start_excel"], group_bounds["data_end_excel"]
    gap_start, gap_end = gap_bounds["data_start_excel"], gap_bounds["data_end_excel"]
    src_start, src_end = source_bounds["data_start_excel"], source_bounds["data_end_excel"]
    kpi_specs = [
        ((3, 0, 3, 1), (4, 0, 5, 1), "Listing hạt giống đối thủ", f"=SUM('Nhóm thị trường'!$C${group_start}:$C${group_end})", as_number(data["qa"].get("competitor_seed_products"))),
        ((3, 3, 3, 4), (4, 3, 5, 4), "Listing hạt giống Hiệp Thủy", f'=COUNTIF(\'SKU đã phân loại\'!$A${src_start}:$A${src_end},"Hiệp Thủy")', as_number(data["qa"].get("hiep_thuy_seed_products"))),
        ((3, 6, 3, 7), (4, 6, 5, 7), "Nhóm nhu cầu cao", f'=COUNTIF(\'Nhóm thị trường\'!$M${group_start}:$M${group_end},"Cao")', sum(1 for row in data["group_summary"] if row.get("demand_tier") == "Cao")),
        ((3, 9, 3, 10), (4, 9, 5, 10), "Cụm Hiệp Thủy chưa có", f'=COUNTIF(\'Khoảng trống HT\'!$L${gap_start}:$L${gap_end},"Chưa có")', sum(1 for row in data["cluster_gap"] if row.get("ht_status") == "Chưa có")),
    ]
    for label_range, value_range, label, formula, cached in kpi_specs:
        sheet.merge_range(*label_range, label, fmt["kpi_label"])
        sheet.merge_range(*value_range, "", fmt["kpi_value"])
        sheet.write_formula(value_range[0], value_range[1], formula, fmt["kpi_value"], cached)
    sheet.merge_range(7, 0, 7, 6, "Nhóm phổ biến và nhu cầu cao", fmt["section"])
    top_headers = ["Hạng", "Nhóm", "Khoảng giá phổ biến", "Giá median", "Sold count", "Số review", "Mức nhu cầu"]
    for col, header in enumerate(top_headers):
        sheet.write(8, col, header, fmt["header"])
    for index, record in enumerate(data["group_summary"][:8]):
        row = 9 + index
        values = [record.get("demand_rank"), record.get("market_group"), price_band(record), as_number(record.get("price_median_vnd")), as_number(record.get("sold_count")), as_number(record.get("review_count")), record.get("demand_tier")]
        for col, value in enumerate(values):
            cell_format = fmt["money"] if col == 3 else fmt["integer"] if col in (0, 4, 5) else fmt["wrap"] if col in (1, 2) else None
            sheet.write(row, col, value, cell_format)
    add_table(sheet, 8, 0, 16, 6, "TopMarketGroupsTable", top_headers)

    group_by_name = {row["market_group"]: row for row in data["group_summary"]}
    cluster_by_name = {row["product_cluster"]: row for row in data["cluster_gap"]}
    ht_group_counts: dict[str, int] = {}
    for row in data["source_seed"]:
        if row.get("shop_name") == "Hiệp Thủy":
            group = row.get("market_group", "")
            ht_group_counts[group] = ht_group_counts.get(group, 0) + 1
    specs = [
        ("A1", "Listing tổng hợp / nhiều lựa chọn", cluster_by_name.get("Listing tổng hợp / nhiều lựa chọn", {}), None, "Tạo 3–4 listing nhiều lựa chọn theo nhu cầu: rau thơm, rau cải, rau ăn quả và hoa.", "Hiệp Thủy chưa có; tín hiệu sold/review đứng đầu các cụm.", "A/B test ảnh bìa, số biến thể và tỷ lệ chuyển đổi; không gọi đây là bundle nếu chỉ là listing nhiều variant."),
        ("A2", "Hoa & cây cảnh mở rộng", group_by_name.get("Hoa & cây cảnh", {}), "Có nhưng còn mỏng", "Mở rộng dừa cạn, mười giờ, hoa mùa vụ/Tết; gom theo mùa và màu.", "32,8 nghìn sold và 4,8 nghìn review; danh mục HT có nhưng còn ít chiều sâu.", "Chạy thử theo mùa; tách evergreen khỏi hoa Tết để tránh tồn kho."),
        ("A3", "Cà chua", cluster_by_name.get("Cà chua", {}), None, "Thêm cà chua bi đỏ/vàng, cherry và giống chịu nhiệt.", "27 listing đối thủ, hơn 11 nghìn sold; HT mới có 4 listing.", "Ưu tiên 3–5 SKU có khác biệt rõ về màu, kích thước và khí hậu."),
        ("B1", "Dưa leo", cluster_by_name.get("Dưa leo", {}), None, "Bổ sung dưa leo Nhật, baby và giống chịu nhiệt.", "Có mặt ở cả 3 shop; HT chỉ có 1 listing trong taxonomy.", "Giữ giá thử nghiệm gần dải phổ biến, tránh SKU gói lớn làm lệch giá."),
        ("B2", "Hành & hẹ", cluster_by_name.get("Hành & hẹ", {}), None, "Mở rộng hành hương/chịu nhiệt và hẹ ta.", "9,0 nghìn sold và 1,4 nghìn review; HT mới có 2 listing.", "Theo dõi tỷ lệ nảy mầm và review 1–3 sao sau lô đầu."),
        ("B3", "Rau củ khác", cluster_by_name.get("Rau củ khác", {}), None, "Test củ đậu/củ sắn và củ dền với 3–5 SKU.", "Hiệp Thủy chưa có; hiện diện ở cả 3 shop benchmark.", "Nhu cầu thấp hơn nhóm lõi; test nhỏ trước khi mở rộng."),
        ("C1", "Rau mầm đậu/ngũ cốc", cluster_by_name.get("Rau mầm – đậu/ngũ cốc", {}), None, "Thử đậu Hà Lan, đậu xanh hoặc ngũ cốc mầm theo pack nhỏ.", "HT chưa có cụm này; có tín hiệu ở 2 shop nhưng quy mô còn nhỏ.", "Chỉ mở rộng nếu sell-through và repeat purchase đạt ngưỡng nội bộ."),
    ]
    sheet.merge_range(18, 0, 18, 15, "Danh mục tiềm năng để thử nghiệm hoặc mở rộng", fmt["section"])
    headers = ["Ưu tiên", "Cơ hội", "Trạng thái HT", "Tín hiệu nhu cầu", "Khoảng giá phổ biến", "Đề xuất hành động", "Vì sao", "Điều kiện kiểm soát"]
    sheet.write(19, 0, headers[0], fmt["header"])
    sheet.write(19, 1, headers[1], fmt["header"])
    sheet.write(19, 2, headers[2], fmt["header"])
    sheet.write(19, 3, headers[3], fmt["header"])
    sheet.write(19, 4, headers[4], fmt["header"])
    sheet.merge_range(19, 5, 19, 7, headers[5], fmt["header"])
    sheet.merge_range(19, 8, 19, 10, headers[6], fmt["header"])
    sheet.merge_range(19, 11, 19, 15, headers[7], fmt["header"])
    for index, (priority, label, record, forced_status, action, why, guardrail) in enumerate(specs):
        row = 20 + index
        status = forced_status or record.get("ht_status") or ("Chưa có" if ht_group_counts.get(label, 0) == 0 else "Có nhưng còn mỏng")
        signal = f"{js_round(record.get('sold_count')):,} sold | {js_round(record.get('review_count')):,} review"
        sheet.write(row, 0, priority, fmt["priority"])
        sheet.write(row, 1, label, fmt["body_border"])
        sheet.write(row, 2, status, fmt["body_border"])
        sheet.write(row, 3, signal, fmt["body_border"])
        sheet.write(row, 4, price_band(record), fmt["body_border"])
        sheet.merge_range(row, 5, row, 7, action, fmt["body_border"])
        sheet.merge_range(row, 8, row, 10, why, fmt["body_border"])
        sheet.merge_range(row, 11, row, 15, guardrail, fmt["body_border"])
        sheet.set_row(row, 56)
    sheet.conditional_format(20, 2, 26, 2, {"type": "text", "criteria": "containing", "value": "Chưa có", "format": workbook.add_format({"bg_color": COLORS["red_light"], "font_color": COLORS["red"], "bold": True})})
    sheet.conditional_format(20, 2, 26, 2, {"type": "text", "criteria": "containing", "value": "Có nhưng còn mỏng", "format": workbook.add_format({"bg_color": COLORS["amber_light"], "font_color": "#8A5A00"})})

    sheet.merge_range(28, 0, 28, 15, "Kết luận sử dụng: ưu tiên test nhỏ, theo dõi sell-through, conversion và chất lượng review. Không xem sold_count tích lũy là forecast doanh số.", fmt["blue_note"])
    sheet.write(29, 0, "Nhóm", fmt["helper"])
    sheet.write(29, 1, "Sold count", fmt["helper"])
    for index, record in enumerate(data["group_summary"][:8]):
        row = 30 + index
        source_excel_row = group_start + index
        sheet.write_formula(row, 0, f"='Nhóm thị trường'!B{source_excel_row}", fmt["helper"], record.get("market_group", ""))
        sheet.write_formula(row, 1, f"='Nhóm thị trường'!H{source_excel_row}", fmt["integer"], as_number(record.get("sold_count")))
    chart = workbook.add_chart({"type": "bar"})
    chart.add_series(
        {
            "name": "Sold count",
            "categories": "='Tóm tắt'!$A$31:$A$38",
            "values": "='Tóm tắt'!$B$31:$B$38",
            "fill": {"color": COLORS["green2"]},
            "border": {"none": True},
        }
    )
    chart.set_title({"name": "Sold count theo nhóm — top 8", "name_font": {"size": 12}})
    chart.set_legend({"none": True})
    chart.set_x_axis({"num_format": "#,##0", "min": 0})
    chart.set_y_axis({"label_position": "low"})
    chart.set_style(10)
    sheet.insert_chart("I8", chart, {"x_scale": 1.20, "y_scale": 1.15})
    sheet.hide_gridlines(2)
    sheet.freeze_panes(2, 0)
    widths = [11, 29, 20, 24, 22, 15, 15, 15, 15, 15, 15, 13, 13, 13, 13, 13]
    for col, width in enumerate(widths):
        sheet.set_column(col, col, width)


def build_method_sheet(sheet, data: dict, fmt: dict) -> None:
    write_title(sheet, 7, "PHƯƠNG PHÁP, PHẠM VI VÀ GIỚI HẠN", fmt)
    method_rows = [
        ("Mục", "Nội dung"),
        ("Phạm vi thị trường", "Ba shop benchmark trên Shopee: Hà Bắc, Quang Anh VNUA và Vườn Babylon. Không tính An Việt hoặc Hiệp Thủy trong nhu cầu/giá; Hiệp Thủy chỉ dùng để đo độ phủ/gap. Không đại diện toàn bộ Shopee."),
        ("Định nghĩa listing hạt giống", "Tên hoặc danh mục có tín hiệu hạt giống; loại phụ kiện/đầu vào như giá thể, khay ươm, viên nén, xơ dừa, phân bón, kích rễ và quà tặng hạt giống."),
        ("Giá đại diện", "(price_min_vnd + price_max_vnd) / 2 khi cả hai có dữ liệu; nếu chỉ có một mức thì dùng mức có sẵn."),
        ("Khoảng giá phổ biến", "P25–P75 của giá đại diện theo listing; median được báo riêng. Đây là dải trung tâm 50%, không phải min–max."),
        ("Nhu cầu", "Sold count và rating_count tích lũy tại snapshot crawl. Chỉ số nhu cầu = trung bình percentile của sold count và số review."),
        ("Điểm cơ hội", "50% chỉ số nhu cầu + 15% độ phủ shop + 20% khoảng trống danh mục + 15% độ rộng bằng chứng. Độ rộng bằng chứng xét số listing và mức tập trung Top 1 sold."),
        ("Trạng thái Hiệp Thủy", "Chưa có = 0 listing trong cụm. Có nhưng còn mỏng = số listing HT thấp hơn đáng kể mức trung bình của ba đối thủ. Đã phủ tương đối = đạt hoặc gần benchmark."),
        ("Giới hạn sold count", "Shopee có thể hiển thị số làm tròn như 1k+/10k+; số liệu không có cửa sổ thời gian chung và không phải doanh thu."),
        ("Giới hạn listing nhiều lựa chọn", "Một listing có thể gom nhiều variant; sold/review được tính ở cấp listing, không tách theo từng giống cụ thể. Vì vậy cần A/B test trước khi nhân rộng."),
        ("Ngày phân tích", data.get("analysis_date", "")),
        ("QA", f"{data['qa'].get('competitor_seed_products', 0)} listing từ đúng 3 shop benchmark; {data['qa'].get('hiep_thuy_seed_products', 0)} listing Hiệp Thủy chỉ dùng cho gap; An Việt bị loại hoàn toàn; {data['qa'].get('unclassified_competitor_products', 0)} dòng chưa phân loại; {data['qa'].get('duplicate_shop_item', 0)} duplicate shop + item_id; {data['qa'].get('missing_price_competitor_products', 0)} listing đối thủ thiếu giá."),
    ]
    start_row = 2
    for index, (label, body) in enumerate(method_rows):
        row = start_row + index
        if index == 0:
            sheet.write(row, 0, label, fmt["header"])
            sheet.merge_range(row, 1, row, 7, body, fmt["header"])
            sheet.set_row(row, 32)
        else:
            sheet.write(row, 0, label, fmt["method_label"])
            sheet.merge_range(row, 1, row, 7, body, fmt["method_body"])
            sheet.set_row(row, 42)
    sheet.set_column(0, 0, 28)
    sheet.set_column(1, 7, 18)
    sheet.hide_gridlines(2)
    sheet.freeze_panes(3, 0)


def build_workbook(analysis_json: Path, output_xlsx: Path) -> dict:
    data = json.loads(analysis_json.read_text(encoding="utf-8"))
    output_xlsx.parent.mkdir(parents=True, exist_ok=True)
    workbook = xlsxwriter.Workbook(
        output_xlsx,
        {"constant_memory": False, "strings_to_formulas": False, "strings_to_urls": False},
    )
    workbook.set_properties(
        {
            "title": "MIS495 Seed Market Gap Analysis",
            "subject": "Shopee seed market benchmark and Hiệp Thủy portfolio gaps",
            "author": "Thy Nguyễn",
            "comments": "Generated reproducibly from four Shopee product CSV files.",
        }
    )
    fmt = build_formats(workbook)
    summary = workbook.add_worksheet("Tóm tắt")
    group = workbook.add_worksheet("Nhóm thị trường")
    gap = workbook.add_worksheet("Khoảng trống HT")
    source = workbook.add_worksheet("SKU đã phân loại")
    method = workbook.add_worksheet("Phương pháp")
    source_bounds = build_source_sheet(workbook, source, data, fmt)
    group_bounds = build_group_sheet(workbook, group, data, fmt, source_bounds)
    gap_bounds = build_gap_sheet(workbook, gap, data, fmt, source_bounds)
    build_summary_sheet(workbook, summary, data, fmt, group_bounds, gap_bounds, source_bounds)
    build_method_sheet(method, data, fmt)
    workbook.close()
    return {
        "output": str(output_xlsx),
        "sheets": ["Tóm tắt", "Nhóm thị trường", "Khoảng trống HT", "SKU đã phân loại", "Phương pháp"],
        "source_rows": len(data["source_seed"]),
        "group_rows": len(data["group_summary"]),
        "gap_rows": len(data["cluster_gap"]),
    }


def main() -> None:
    parser = argparse.ArgumentParser(description="Tạo workbook MIS495 trên Google Colab bằng XlsxWriter.")
    parser.add_argument("analysis_json", type=Path)
    parser.add_argument("output_xlsx", type=Path)
    args = parser.parse_args()
    result = build_workbook(args.analysis_json, args.output_xlsx)
    print(json.dumps(result, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()


## 7. Tạo workbook và kiểm tra cấu trúc/công thức

In [ ]:
OUTPUT_XLSX = OUTPUT_DIR / OUTPUT_FILENAME
build_command = [
    sys.executable,
    str(Path.cwd() / "build_seed_market_workbook_colab.py"),
    str(ANALYSIS_JSON),
    str(OUTPUT_XLSX),
]
build_run = subprocess.run(build_command, check=True, text=True, capture_output=True)
print(build_run.stdout)

expected_sheets = ["Tóm tắt", "Nhóm thị trường", "Khoảng trống HT", "SKU đã phân loại", "Phương pháp"]
expected_dimensions = {
    "Tóm tắt": (38, 16),
    "Nhóm thị trường": (26, 15),
    "Khoảng trống HT": (53, 19),
    "SKU đã phân loại": (794, 12),
    "Phương pháp": (14, 8),
}

formula_book = openpyxl.load_workbook(OUTPUT_XLSX, data_only=False, read_only=False)
value_book = openpyxl.load_workbook(OUTPUT_XLSX, data_only=True, read_only=False)
workbook_checks = []
workbook_checks.append({"check": "sheet_order", "expected": expected_sheets, "actual": formula_book.sheetnames, "pass": formula_book.sheetnames == expected_sheets})
for sheet_name, expected_shape in expected_dimensions.items():
    sheet = formula_book[sheet_name]
    actual_shape = (sheet.max_row, sheet.max_column)
    workbook_checks.append({"check": f"dimension::{sheet_name}", "expected": expected_shape, "actual": actual_shape, "pass": actual_shape == expected_shape})

formula_count = sum(
    1
    for sheet in formula_book.worksheets
    for row in sheet.iter_rows()
    for cell in row
    if isinstance(cell.value, str) and cell.value.startswith("=")
)
formula_errors = []
error_tokens = {"#REF!", "#DIV/0!", "#VALUE!", "#NAME?", "#N/A"}
for sheet in value_book.worksheets:
    for row in sheet.iter_rows():
        for cell in row:
            if isinstance(cell.value, str) and cell.value in error_tokens:
                formula_errors.append(f"{sheet.title}!{cell.coordinate}: {cell.value}")
workbook_checks.extend(
    [
        {"check": "formula_cells_present", "expected": "> 0", "actual": formula_count, "pass": formula_count > 0},
        {"check": "formula_error_cells", "expected": 0, "actual": len(formula_errors), "pass": len(formula_errors) == 0},
        {"check": "output_file_nonempty", "expected": True, "actual": OUTPUT_XLSX.stat().st_size > 0, "pass": OUTPUT_XLSX.stat().st_size > 0},
    ]
)
workbook_qa = pd.DataFrame(workbook_checks)
display(workbook_qa)
assert workbook_qa["pass"].all(), "WORKBOOK QA FAILED — xem bảng kiểm tra phía trên"
print("FINAL WORKBOOK QA: PASS")
print("Output:", OUTPUT_XLSX)


## 8. Đối chiếu với file Excel tham chiếu — tùy chọn

Nếu đã upload file Excel kết quả cùng dữ liệu đầu vào, cell này so sánh:

- Thứ tự/tên sheet.
- Số hàng và số cột đã sử dụng.
- Giá trị hiển thị của từng ô, với tolerance số học `1e-8`.

So sánh công thức không bắt buộc vì notebook Colab dùng một engine xuất Excel khác; tiêu chí chính là kết quả và cấu trúc có khớp hay không.


In [ ]:
reference_comparison = {"status": "SKIPPED", "difference_count": None, "differences_preview": []}


def values_equal(left, right, tolerance=1e-8):
    if isinstance(left, (int, float)) and isinstance(right, (int, float)):
        return abs(float(left) - float(right)) <= tolerance
    return left == right


if reference_xlsx:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        reference_book = openpyxl.load_workbook(reference_xlsx, data_only=True, read_only=False)
        generated_book = openpyxl.load_workbook(OUTPUT_XLSX, data_only=True, read_only=False)
    differences = []
    if reference_book.sheetnames != generated_book.sheetnames:
        differences.append({"type": "sheet_order", "expected": reference_book.sheetnames, "actual": generated_book.sheetnames})
    for sheet_name in reference_book.sheetnames:
        if sheet_name not in generated_book.sheetnames:
            continue
        expected_sheet = reference_book[sheet_name]
        actual_sheet = generated_book[sheet_name]
        if (expected_sheet.max_row, expected_sheet.max_column) != (actual_sheet.max_row, actual_sheet.max_column):
            differences.append(
                {
                    "type": "dimension",
                    "sheet": sheet_name,
                    "expected": (expected_sheet.max_row, expected_sheet.max_column),
                    "actual": (actual_sheet.max_row, actual_sheet.max_column),
                }
            )
        for row in range(1, max(expected_sheet.max_row, actual_sheet.max_row) + 1):
            for col in range(1, max(expected_sheet.max_column, actual_sheet.max_column) + 1):
                expected_value = expected_sheet.cell(row, col).value
                actual_value = actual_sheet.cell(row, col).value
                if not values_equal(expected_value, actual_value):
                    differences.append(
                        {
                            "type": "cell",
                            "sheet": sheet_name,
                            "cell": actual_sheet.cell(row, col).coordinate,
                            "expected": expected_value,
                            "actual": actual_value,
                        }
                    )
    reference_comparison = {
        "status": "PASS" if not differences else "FAIL",
        "difference_count": len(differences),
        "differences_preview": differences[:20],
    }
    print("REFERENCE COMPARISON:", reference_comparison["status"])
    print("Differences:", reference_comparison["difference_count"])
    if differences:
        display(pd.DataFrame(differences[:20]))
    assert not differences, "REFERENCE COMPARISON FAILED — xem tối đa 20 khác biệt đầu tiên phía trên"
else:
    print("REFERENCE COMPARISON: SKIPPED — không có file .xlsx tham chiếu được upload")


## 9. Xuất báo cáo QA và tải file kết quả

In [ ]:
qa_report = {
    "analysis_date": ANALYSIS_DATE,
    "input_files": [str(INPUT_DIR / name) for name in REQUIRED_FILENAMES],
    "data_qa_pass": bool(qa_table["pass"].all()),
    "workbook_qa_pass": bool(workbook_qa["pass"].all()),
    "reference_comparison": reference_comparison,
    "output_xlsx": str(OUTPUT_XLSX),
}
QA_REPORT = OUTPUT_DIR / "MIS495_Seed_Market_Gap_QA_Report.json"
QA_REPORT.write_text(json.dumps(qa_report, ensure_ascii=False, indent=2), encoding="utf-8")

print(json.dumps(qa_report, ensure_ascii=False, indent=2))
print("\nFINAL QA: PASS")

if in_colab:
    colab_files.download(str(OUTPUT_XLSX))
    colab_files.download(str(QA_REPORT))
else:
    print("Workbook:", OUTPUT_XLSX)
    print("QA report:", QA_REPORT)


## 10. Cách đọc kết quả

- `Tóm tắt`: kết luận chính, nhóm nhu cầu cao và danh mục đề xuất.
- `Nhóm thị trường`: 17 nhóm, dải giá P25–P75 và sold/review.
- `Khoảng trống HT`: 50 cụm, trạng thái Hiệp Thủy và điểm cơ hội.
- `SKU đã phân loại`: 791 listing dùng để truy vết.
- `Phương pháp`: phạm vi, công thức và giới hạn dữ liệu.

`sold_count` và số review là số tích lũy tại snapshot crawl, không phải doanh thu hoặc dự báo doanh số.
